# 1_1. Survival analysis per tooth

idea: make one model per tooth

In [1]:
import lifelines
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
import optuna
from lifelines import WeibullAFTFitter, LogLogisticAFTFitter, LogNormalAFTFitter

c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [2]:
deciduous_teeth = pd.read_csv("../data/deciduous_teeth.csv", sep=";")

In [3]:
permanent_teeth = pd.read_csv("../data/permanent_teeth.csv", sep=";")

## Loop over all teeth

### Preprocess

In [4]:
def preprocess_data(df):
    df.loc[((df["LOWER"].isna()), "LOWER")] = 0
    df.loc[((df["UPPER"].isna()), "UPPER")] = np.inf

    # fillna bc linear aft can't handle missing data
    df = df.fillna("unknown")

    return df

In [5]:
deciduous_teeth = preprocess_data(deciduous_teeth)
permanent_teeth = preprocess_data(permanent_teeth)

In [6]:
deciduous_teeth["SEX"].value_counts()

SEX
F          28224
M          27272
unknown      560
Name: count, dtype: int64

### nested CV per tooth

In [7]:
feature_combos = [
    "DEVELOPM_STAGE+ C(SEX)+ C(BREED_SIZE)+ C(SKULL_TYPE)+ C(BREED_CLADE)",
    "DEVELOPM_STAGE+ C(BREED_SIZE)+ C(SKULL_TYPE)+ C(BREED_CLADE)",
    "DEVELOPM_STAGE+ C(SEX)+ C(BREED_SIZE)+ C(SKULL_TYPE)",
    "DEVELOPM_STAGE+ C(BREED_SIZE)+ C(SKULL_TYPE)",
]

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3

In [8]:
def nested_cv(tooth_df, df_preds, times):
    lodo = StratifiedGroupKFold(N_OUTER_FOLDS, shuffle=True, random_state=42)

    for i, (train_index, test_index) in enumerate(
        lodo.split(X=tooth_df, y=tooth_df["CENS"], groups=tooth_df["DOG"])
    ):
        print(i)
        train_outer = tooth_df.iloc[train_index]
        test_outer = tooth_df.iloc[test_index]

        # INNER LOOP: hyperparam tuning
        def objective(trial):
            feature_selection = trial.suggest_int(
                "feature_selection", 0, len(feature_combos) - 1
            )
            fitter = trial.suggest_categorical(
                "fitter",
                ["WeibullAFTFitter", "LogLogisticAFTFitter"],
            )

            params = {
                "penalizer": trial.suggest_float("penalizer", 1e-6, 1e-1, log=True),
                "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
            }

            CV_scores = []

            train_events = tooth_df.loc[train_index, "CENS"]
            train_ids = tooth_df.loc[train_index, "DOG"]
            sgkf = StratifiedGroupKFold(
                n_splits=N_INNER_FOLDS, shuffle=True, random_state=42
            )
            for idx, (train_idx_inner, test_idx_inner) in enumerate(
                sgkf.split(train_outer, train_events, train_ids)
            ):
                train_inner, test_inner = (
                    train_outer.iloc[train_idx_inner],
                    train_outer.iloc[test_idx_inner],
                )

                inner_fitter = eval(fitter)(**params)
                try:
                    inner_fitter.fit_interval_censoring(
                        train_inner,
                        "LOWER",
                        "UPPER",
                        formula=feature_combos[feature_selection],
                    )

                    CV_scores.append(
                        inner_fitter.score(test_inner, scoring_method="log_likelihood")
                    )
                except:
                    continue

            return np.mean(CV_scores)

        # Optuna for hyperparam tuning
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=10, show_progress_bar=False)
        best_params = {}
        best_params.update(study.best_trial.params)

        # get best params out of inner loop
        best_features = best_params.pop("feature_selection")
        best_fitter = best_params.pop("fitter")

        # train model on best params w/ train of outer loop
        fitter_outer = eval(best_fitter)(**best_params)
        fitter_outer.fit_interval_censoring(
            train_outer, "LOWER", "UPPER", formula=feature_combos[best_features]
        )

        # Predict survival probabilities for all timepoints
        sf = fitter_outer.predict_survival_function(test_outer, times)

        # Convert to long-format DataFrame
        sf_df = sf.reset_index(  # each row = a timepoint, each column = an individual
            names="time"
        ).melt(id_vars="time", var_name="index_in_test", value_name="pred_survival")

        # Add identifying columns from test_outer
        meta = (
            test_outer.reset_index()
            .loc[:, ["index", "DOG", "TOOTH_NUMBER", "DEVELOPM_STAGE"]]
            .rename(columns={"index": "index_in_test"})
        )

        # Merge predictions with metadata
        fold_preds = sf_df.merge(meta, on="index_in_test")

        # Store in list
        df_preds = pd.concat([df_preds, fold_preds], ignore_index=True)
    return df_preds

In [ ]:
def run_for_all_teeth(df):
    df_preds = pd.DataFrame(
        columns=[
            "time",
            "index_in_test",
            "pred_survival",
            "DOG",
            "TOOTH_NUMBER",
            "DEVELOPM_STAGE",
        ]
    )

    times = df["LOWER"].unique()
    times = np.append(times, df["UPPER"].unique())
    times = np.unique(times)
    times = np.delete(times, np.where(times == np.inf))

    for tooth in df["TOOTH_NUMBER"].unique():
        print(f"Starting nested CV for tooth {tooth}")

        tooth_df = df[df["TOOTH_NUMBER"] == tooth].reset_index(drop=True)

        while True:
            try:
                df_preds = nested_cv(tooth_df, df_preds, times)
                break
            except:  # rerun if convergence error
                print(f"Retrying for tooth {tooth}")
                df_preds = df_preds[df_preds["TOOTH_NUMBER"] != tooth]

    return df_preds

In [ ]:
df_preds_dec = run_for_all_teeth(deciduous_teeth)
df_preds_perma = run_for_all_teeth(permanent_teeth)

In [ ]:
df_preds_dec.to_csv(r"preds/ll_tooth_preds_dec.csv", sep=";")
df_preds_perma.to_csv(r"preds/ll_tooth_preds_perma.csv", sep=";")